<a href="https://colab.research.google.com/github/hanidew/G12_IR_Evaluation/blob/main/IRWS_G12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Cleaning (Hani)

50 topics each topic got 1000 documents

In [15]:
import pandas as pd
import glob
import os
import csv

print("1. Loading System Runs...")
run_cols = ['Topic_ID', 'Iteration', 'Doc_ID', 'Rank', 'Score', 'Trash']

# UPDATE THIS PATH to your folder containing the 15 system runs
run_files_path = '/content/drive/MyDrive/IRWS_G12/set12/System_Runs/*'

all_runs_list = []
for file in glob.glob(run_files_path):
    # Using our bulletproof parser
    df = pd.read_csv(file, sep=r'\s+', names=run_cols, quoting=csv.QUOTE_NONE, na_filter=False)

    # Extract the true system name from the filename
    df['System_Name'] = os.path.basename(file).split('.')[-1]

    all_runs_list.append(df)

# Combine into one DataFrame
full_runs_df = pd.concat(all_runs_list, ignore_index=True)

# Drop the broken internal system name column and ensure Topic_ID is a string
full_runs_df.drop(columns=['Trash'], inplace=True)
full_runs_df['Topic_ID'] = full_runs_df['Topic_ID'].astype(str)

print("Data successfully loaded!")

print("\n2. Running Depth Audit...")
# Group by System and Topic to count documents
depth_audit = full_runs_df.groupby(['System_Name', 'Topic_ID'])['Doc_ID'].count().reset_index()
depth_audit.rename(columns={'Doc_ID': 'Document_Count'}, inplace=True)

# Filter for any anomalies
anomalies = depth_audit[depth_audit['Document_Count'] < 1000]

print(f"Total system/topic combinations checked: {len(depth_audit)}")
print(f"Total anomalies found (< 1000 docs): {len(anomalies)}")

if len(anomalies) > 0:
    print("\nAnomalies are still present in the new files. Here they are:")
    print(anomalies)
else:
    print("\nPERFECT RUN! All 15 systems successfully retrieved exactly 1000 documents per topic!")

1. Loading System Runs...
Data successfully loaded!

2. Running Depth Audit...
Total system/topic combinations checked: 750
Total anomalies found (< 1000 docs): 0

PERFECT RUN! All 15 systems successfully retrieved exactly 1000 documents per topic!


In [14]:
# --- THE DATA HANDOFF ---
# Define the path where you want to save the clean data in your Shared Drive
# Make sure this folder is accessible to all 5 members!
export_path = '/content/drive/MyDrive/IRWS_G12/set12/clean_system_runs_v1.csv'

# Export the dataframe to a CSV file.
# index=False prevents pandas from adding an extra column of row numbers
full_runs_df.to_csv(export_path, index=False)

print(f"Data successfully exported to: {export_path}")
print("Members 2 and 3 can now load this file!")

Data successfully exported to: /content/drive/MyDrive/IRWS_G12/set12/clean_system_runs_v1.csv
Members 2 and 3 can now load this file!
